In [ ]:
""" 
Tarık Buğra Ay - 042101100
"""
#--------------------------------------------------------------
!pip install -q librosa tqdm scikit-learn gdown tensorflow keras
#--------------------------------------------------------------
import os, shutil, zipfile
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import load_model
import gdown


SR = 22050
N_MELS = 256
FMAX = 8000
FIXED_TIME_STEPS = 150
BATCH_SIZE = 64

#  Paths -----------------------------------------------------------------------------
SECRET_AUDIO_DIR = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public"
SECRET_CSV_PATH = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public.csv"
#--------------------------------------------------------------------------------------------


GDRIVE_ZIP_URL = "https://drive.google.com/uc?export=download&id=1e7tAfF72Qii8WcEs6GovKJ_UVrhjBE_J"
MODEL_ZIP_PATH = "/kaggle/working/models_from_drive.zip"
TEMP_MODEL_DIR = "/kaggle/working/_temp_models"
TEMP_CACHE_DIR = "/kaggle/working/_temp_cache"

os.makedirs(TEMP_MODEL_DIR, exist_ok=True)
os.makedirs(TEMP_CACHE_DIR, exist_ok=True)

# Download Model ZIP 
print(" Downloading model ZIP from Google Drive...")
gdown.download(GDRIVE_ZIP_URL, MODEL_ZIP_PATH, quiet=False)

# Unzip 
with zipfile.ZipFile(MODEL_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(TEMP_MODEL_DIR)
print(" Models unzipped.\n")

# Load Metadata 
df_test = pd.read_csv(SECRET_CSV_PATH)
df_test["filepath"] = df_test["file_name"].apply(lambda x: os.path.join(SECRET_AUDIO_DIR, x))

# Cache Mels 
def cache_test_mels(df, cache_dir):
    mel_paths = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=" Caching mels"):
        wav_path = row["filepath"]
        mel_path = os.path.join(cache_dir, os.path.basename(wav_path).replace(".wav", "_mel.npy"))
        mel_paths.append(mel_path)
        if os.path.exists(mel_path): continue
        try:
            y, sr = librosa.load(wav_path, sr=SR)
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
            mel_db = librosa.util.fix_length(mel_db, size=FIXED_TIME_STEPS, axis=1)
            mel_db = np.expand_dims(mel_db, axis=-1)
            np.save(mel_path, mel_db)
        except Exception as e:
            print(f"[ERROR] {wav_path}: {e}")
            mel_paths[-1] = None
    df["mel_path"] = mel_paths
    return df[df["mel_path"].notnull()]

df_test = cache_test_mels(df_test, TEMP_CACHE_DIR)
X_test = np.array([np.load(p) for p in df_test["mel_path"]])
y_test = df_test["classID"].values
print(f"\nLoaded {len(X_test)} mel samples.")

#  Load Models 
models, model_names = [], []
for root, _, files in os.walk(TEMP_MODEL_DIR): 
    for fname in sorted(files):
        if fname.endswith(".keras"):
            fpath = os.path.join(root, fname)
            try:
                models.append(load_model(fpath))
                model_names.append(fname)
                print(f"  Loaded: {fname}")
            except Exception as e:
                print(f"  Failed to load {fname}: {e}")

# Evaluate Individual Models 
all_probs = []
model_scores = {}
fold_accuracies = []

print("\n*** Individual Model Evaluations ***")
for i, model in enumerate(models):
    print(f"\n Evaluating {model_names[i]}")
    try:
        probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    except Exception as e:
        print(f"  Model {model_names[i]} crashed: {e}")
        continue

    preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds, digits=3)

    print(f"   Accuracy: {acc:.4f}")
    print(report)

    all_probs.append(probs)
    model_scores[model_names[i]] = {"accuracy": acc, "report": report}
    fold_accuracies.append(acc)

#  Soft Voting Ensemble 
avg_probs = np.mean(all_probs, axis=0)
soft_preds = np.argmax(avg_probs, axis=1)
soft_acc = accuracy_score(y_test, soft_preds)
soft_report = classification_report(y_test, soft_preds, digits=3)

model_scores["Ensemble_Soft"] = {"accuracy": soft_acc, "report": soft_report}
print("\n Ensemble (Soft Voting) ")
print(f" Accuracy: {soft_acc:.4f}")
print(soft_report)

#  Weighted Soft Voting Ensemble 
weights = np.array(fold_accuracies)
weights /= np.sum(weights)  # Normalize
weighted_probs = np.tensordot(weights, np.stack(all_probs), axes=1)
weighted_preds = np.argmax(weighted_probs, axis=1)
weighted_acc = accuracy_score(y_test, weighted_preds)
weighted_report = classification_report(y_test, weighted_preds, digits=3)

model_scores["Ensemble_Weighted"] = {"accuracy": weighted_acc, "report": weighted_report}
print("\n Ensemble (Weighted Soft Voting) ")
print(f" Accuracy: {weighted_acc:.4f}")
print(weighted_report)

#  Final Summary 
print("\n  **** Final Model Performance Summary ****")
best_model_name, best_score = max(model_scores.items(), key=lambda x: x[1]["accuracy"])
print(f"\n Best Performing: {best_model_name} — Accuracy: {best_score['accuracy']:.4f}")
print(best_score["report"])

# Cleanup 
shutil.rmtree(TEMP_MODEL_DIR)
shutil.rmtree(TEMP_CACHE_DIR)
os.remove(MODEL_ZIP_PATH)
print("\n Cleanup complete. All temporary files removed.")